In [3]:
from config.paths import POLICIES_DATA_PATH, POLICIES_DIR
from striprtf.striprtf import rtf_to_text
from common.logger import logger
from PyPDF2 import PdfReader
from tqdm import tqdm
import pandas as pd
import datetime
import json
import re

df = pd.read_csv(POLICIES_DATA_PATH)
hv = []
logger.info(f"Storing {df.shape[0]} policies into the database.")
all_topics = set()
cleaned_df = pd.DataFrame()

for _, row in df.iterrows():
    data = {}
    code = str(row['Código']).strip()
    data["code"] = code
    data["title"] = str(row['Texto depositado']).upper().strip()
    data["type"] = data["code"][:3]

    fecha_deposito = str(row['Fecha Depósito']).strip()
    if fecha_deposito.startswith('Res.'):
        fecha_deposito = fecha_deposito.split('del')[1].strip()

    # Fecha deposito is in format dd/mm/yyyy
    data["deposit_date"] = datetime.date(
        year=int(fecha_deposito.split('/')[2]),
        month=int(fecha_deposito.split('/')[1]),
        day=int(fecha_deposito.split('/')[0])
    )

    linked_policies = str(row['Pólizas/Cláusulas']).strip()
    linked_policies = re.split(r'\n|\s', linked_policies)
    if linked_policies[0] == 'nan':
        linked_policies = []
    data["linked_policies"] = linked_policies

    data["insurance_company"] = str(
        row['Nombre entidad que deposita']).strip()
    data["prohibition_resolution"] = str(row['Res. Prohibición']).strip()
    data["authorization_resolution"] = str(row['Res. Autoriza']).strip()

    is_prohibited = False
    if data["prohibition_resolution"] != "-" and data["prohibition_resolution"] != "nan" and data["prohibition_resolution"] != "":
        is_prohibited = True
    data["is_prohibited"] = is_prohibited

    topics = str(row['Temas']).strip().split('-')
    topics = [topic.strip() for topic in topics if topic != '']
    all_topics.update(topics)
    data["topics"] = json.dumps(topics)
    print(data)
    hv.append(data)
    


(2024-06-27 21:26:08,585) - INFO: Storing 6715 policies into the database.


{'code': 'POL120240099', 'title': 'PÓLIZA DE GARANTÍA DE CORREDOR DE SEGUROS', 'type': 'POL', 'deposit_date': datetime.date(2024, 5, 22), 'linked_policies': [], 'insurance_company': 'CORREDORA DE SEGUROS BRISTOL SPA', 'prohibition_resolution': '-', 'authorization_resolution': '-', 'is_prohibited': False, 'topics': '["Varios Seg. Generales"]'}
{'code': 'POL120240098', 'title': 'PÓLIZA DE SEGURO DE CAUCIÓN PARA VEEDORES Y LIQUIDADORES DE LA LEY DE REORGANIZACIÓN Y LIQUIDACION DE ACTIVOS DE EMPRESAS Y PERSONAS', 'type': 'POL', 'deposit_date': datetime.date(2024, 5, 20), 'linked_policies': [], 'insurance_company': 'ASEGURADORA PORVENIR S.A.', 'prohibition_resolution': '-', 'authorization_resolution': '-', 'is_prohibited': False, 'topics': '["Garant\\u00edas y Fidelidad"]'}
{'code': 'POL320240097', 'title': 'SEGURO COLECTIVO DE REEMBOLSO DE GASTOS MEDICOS POR HOSPITALIZACIÓN', 'type': 'POL', 'deposit_date': datetime.date(2024, 5, 16), 'linked_policies': [], 'insurance_company': 'COLMENA COM

In [9]:
df2 = pd.DataFrame(hv)
df2.to_csv(r"storage\datasets\cleaned_dataset.csv")
df2

,code,title,type,deposit_date,linked_policies,insurance_company,prohibition_resolution,authorization_resolution,is_prohibited,topics
0,POL120240099,PÓLIZA DE GARANTÍA DE CORREDOR DE SEGUROS,POL,2024-05-22,[],CORREDORA DE SEGUROS BRISTOL SPA,-,-,False,"[""Varios Seg. Generales""]"
1,POL120240098,PÓLIZA DE SEGURO DE CAUCIÓN PARA VEEDORES Y LI...,POL,2024-05-20,[],ASEGURADORA PORVENIR S.A.,-,-,False,"[""Garant\u00edas y Fidelidad""]"
2,POL320240097,SEGURO COLECTIVO DE REEMBOLSO DE GASTOS MEDICO...,POL,2024-05-16,[],COLMENA COMPAÑIA DE SEGUROS DE VIDA S.A.,-,-,False,"[""Salud""]"
3,CAD320240096,CLAUSULA ADICIONAL DE INVALIDEZ ACCIDENTAL,CAD,2024-05-14,[POL120130166],BNP PARIBAS CARDIF SEGUROS GENERALES S.A.,-,-,False,"[""Accidentes Personales""]"
4,CAD320240095,CLAUSULA ADICIONAL DE INVALIDEZ PERMANENTE DOS...,CAD,2024-05-14,[POL120130166],BNP PARIBAS CARDIF SEGUROS GENERALES S.A.,-,-,False,"[""Accidentes Personales""]"
...,...,...,...,...,...,...,...,...,...,...
6710,POL288001,POLIZA DE RENTA VITALICIA INMEDIATA (ARTICULO ...,POL,1988-01-25,[],CMF,-,-,False,"[""DEJADA SIN EFECTO"", ""Previsionales""]"
6711,POL288002,POLIZA DE RENTA VITALICIA DIFERIDA (ARTICULO 6...,POL,1988-01-25,[],CMF,-,-,False,"[""DEJADA SIN EFECTO"", ""Previsionales""]"
6712,POL187002,MODELO DE POLIZA DE SEGURO DE RESPONSABILIDAD ...,POL,1987-12-31,[],CMF,-,-,False,"[""Eliminadas""]"
6713,POL287001,MODELO DE POLIZA DE ADMINISTRADORAS DE FONDOS ...,POL,1987-12-03,[],CMF,-,-,False,"[""Eliminadas""]"
